# Ascend Pytorch Profiler 工具使用
本节介绍如何在 PyTorch 框架模型中使用 Ascend PyTorch Profiler 工具分析出需要优化的算子。大纲如下：
- 什么是 Ascend Pytorch Profiler 工具 
- 如何使用 Ascend Pytorch Profiler 工具采集性能数据
- Qwen3-0.8B 性能采集实战
- 性能数据查看与分析

---

## 1 环境准备

本节所有内容均存放于 Sources 文件夹中。
### 1.1 环境初始化
首先对 Jupyter 环境进行初始化。以下代码将环境变量导入 Jupyter 环境，并创建代码目录，以确保 CANN 功能正常使用。

In [ ]:
!mkdir -p Sources/02

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

### 1.2 Python依赖安装
需要安装 **Qwen3-0.8B 性能采集实战** 的相关 Python 依赖：modelscope（用于模型下载）和 transformers（用于模型运行）。

In [ ]:
!pip3 install modelscope transformers==5.3.0 -i https://pypi.tuna.tsinghua.edu.cn/simple -t Sources/02 
import os 
import sys
pkg_dir = os.path.abspath("Sources/02")
if pkg_dir not in sys.path:
    sys.path.insert(0, pkg_dir)

---

##  2 什么是 Ascend Pytorch Profiler 工具 
Ascend PyTorch Profiler 是 Ascend PyTorch 框架提供的一套性能数据采集 API 接口，能够采集框架侧、CANN 侧和 Device 侧的原始性能数据，并完成解析。

---

## 3 如何使用 Ascend Pytorch Profiler 工具采集性能数据
Ascend PyTorch Profiler 支持多种采集方式，本节介绍其中基于接口采集的方式：创建 `torch_npu.profiler.profile` 对象，通过 `start` 和 `stop` 接口控制采集的代码段，`step` 接口划分循环迭代。
```
import torch
import torch_npu
...

# 添加Profiling采集扩展配置参数
experimental_config = torch_npu.profiler._ExperimentalConfig(
    export_type=torch_npu.profiler.ExportType.Text,
    profiler_level=torch_npu.profiler.ProfilerLevel.Level2,
)

# 添加Profiling采集基础配置参数
prof = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.CPU,
        torch_npu.profiler.ProfilerActivity.NPU
        ],
    schedule=torch_npu.profiler.schedule(wait=0, warmup=0, active=1, repeat=1, skip_first=0),
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler("./result"),
    profile_memory=False,
    with_modules=False,
    with_stack=True,
    experimental_config=experimental_config)

prof.start()    # 启动性能数据采集
for step in range(steps):    
    train/infer_one_step()    # 训练/推理函数
    prof.step()    # 与schedule配套使用
prof.stop()    # 结束性能数据采集
```
不同的参数配置会影响工具采集的信息范围。分析需要优化的算子时，通常采用以上参数值即可，但 `schedule` 和 `on_trace_ready` 两个参数经常需要根据实际场景调整。

（1）`schedule` 参数

`schedule` 用于设置不同 step 的采集行为。例如，通过 `schedule` 可以控制上述代码循环中采集哪些 step 的数据。这对于性能分析尤为重要：通常分析算子瓶颈时不希望采集首次推理/训练的耗时，因为首次执行会存在额外的调度开销。此外，在大模型推理场景中，Prefill 推理和 Decode 推理存在于同一代码循环下（首次推理为 Prefill，后续为 Decode），也需要通过该参数指定采集 Prefill 阶段还是 Decode 阶段的推理数据。`schedule` 通过以下参数控制要采集的 step：
| 参数        | 说明 |
|-------------|------|
| wait        | 每次重复执行采集跳过的 step 轮数，int 类型。必选。 |
| active      | 采集的 step 轮数，int 类型。必选。 |
| warmup      | 预热的 step 轮数，int 类型。默认值为 0。建议设置 1 轮预热。可选。 |
| repeat      | 重复执行 wait + warmup + active 的次数，int 类型。取值范围为大于等于 0 的整数，默认值为 0。可选。<br/><br/>**说明：**<br/>当使用 MindStudio Insight 查看性能数据时，建议配置 `repeat = 1`（表示执行 1 次，仅生成一份性能数据），因为：<br/>- repeat > 1 会在同一目录下生成多份性能数据，需要手动拆分文件再分析，按时间戳区分。<br/>- repeat = 0 时，重复执行的具体次数由总训练 step 数决定，例如：<br/>  总 step = 100，wait + active + warmup = 10，skip_first = 10<br/>  则 repeat = (100 - 10) / 10 = 9，表示生成 9 份性能数据。 |
| skip_first  | 采集前先跳过的 step 轮数，int 类型。默认值为 0。动态 Shape 场景建议跳过前 10 轮保证性能数据稳定；其他场景可根据实际情况配置。可选。 |

（2）`on_trace_ready`参数

此参数主要调整 `torch_npu.profiler.tensorboard_trace_handler()` 接口传入的路径字符串，以决定性能数据文件的落盘位置。例如，上面例子中将性能数据存放在 `result` 文件夹中：
```
on_trace_ready=torch_npu.profiler.tensorboard_trace_handler("./result")
```

更多采集方式可参考[《Ascend PyTorch调优工具》](https://gitcode.com/Ascend/pytorch/blob/v2.7.1/docs/zh/ascend_pytorch_profiler/ascend_pytorch_profiler_user_guide.md)

---

## 4 Qwen3-0.8B 性能采集实战
### 4.1 Qwen3-0.8B 模型下载
通过已安装的 modelscope 依赖下载模型 Qwen3.5-0.8B。Qwen3.5-0.8B 是通义千问系列的一个轻量级模型，参数量为 0.8B（8 亿参数），适合快速在 NPU 上部署验证。

以下命令将从ModelScope平台下载模型文件到本地Sources/02/Qwen目录：

In [ ]:
!modelscope download --model Qwen/Qwen3.5-0.8B --local_dir Sources/02/Qwen 

### 4.2 嵌入性能采集代码
大模型采用逐 token 循环推理的方式。当使用 `transformers` 库进行大模型推理时，循环推理的逻辑通常封装在 `generate` 函数中。我们需要采集大模型单次推理的耗时，因此需要定位到循环调用模型的代码位置，即 `generate` 函数的定义位置。

在 `transformers` 库中，`generate` 函数定义在 generation/utils.py 文件中，根据后处理方式的不同会调用不同的函数进行推理和后处理。后处理方式由模型文件中的 `generation_config.json` 定义，或通过推理脚本中 `generate` 参数传入。若模型文件不包含 `generation_config.json` 且调用代码未显式指定后处理方式，则采用默认的 GREEDY_SEARCH（贪心解码）方式。Qwen3.5-0.8B 模型文件不包含 `generation_config.json`，推理脚本中 `generate` 函数指定了 `do_sample` 方式，utils.py 中相应代码段如下（不同 `transformers` 版本可能存在细微差别）：
```
    elif generation_mode in (GenerationMode.SAMPLE, GenerationMode.GREEDY_SEARCH):
        # 11. prepare logits warper
        prepared_logits_warper = (
            self._get_logits_warper(generation_config, device=input_ids.device)
            if generation_config.do_sample
            else None
        )

        # 12. expand input_ids with `num_return_sequences` additional sequences per batch
        input_ids, model_kwargs = self._expand_inputs_for_generation(
            input_ids=input_ids,
            expand_size=generation_config.num_return_sequences,
            is_encoder_decoder=self.config.is_encoder_decoder,
            **model_kwargs,
        )

        # 13. run sample (it degenerates to greedy search when `generation_config.do_sample=False`)
        result = self._sample(
            input_ids,
            logits_processor=prepared_logits_processor,
            logits_warper=prepared_logits_warper,
            stopping_criteria=prepared_stopping_criteria,
            generation_config=generation_config,
            synced_gpus=synced_gpus,
            streamer=streamer,
            **model_kwargs,
        )
```
可以看到真正的推理逻辑定义在`_sample`函数中，它也定义在utils.py里，其中循环推理的代码段如下：
```
while self._has_unfinished_sequences(this_peer_finished, synced_gpus, device=input_ids.device):
    if prefill_consumed:
        next_sequence_length = 1 if model_kwargs["use_cache"] else None
        model_inputs = self.prepare_inputs_for_generation(
            input_ids, next_sequence_length=next_sequence_length, **model_kwargs
        )
        with self._optimize_model_for_decode():
            outputs = model_forward(**model_inputs, return_dict=True)
    prefill_consumed = True
    model_kwargs = self._update_model_kwargs_for_generation(
        outputs,
        model_kwargs,
        is_encoder_decoder=self.config.is_encoder_decoder,
    )
    if synced_gpus and this_peer_finished:
        continue

    # Copy is needed to avoid keeping a hanging ref to outputs.logits which may be very large for first iteration
    # (the clone itself is always small)
    next_token_logits = outputs.logits[:, -1, :].to(copy=True, dtype=torch.float32, device=input_ids.device)

    # pre-process distribution
    next_token_scores = logits_processor(input_ids, next_token_logits)

    # Store scores, attentions and hidden_states when required
    if return_dict_in_generate:
        if output_scores:
            scores += (next_token_scores,)
        if output_logits:
            raw_logits += (next_token_logits,)
        if output_attentions:
            decoder_attentions += (
                (outputs.decoder_attentions,) if self.config.is_encoder_decoder else (outputs.attentions,)
            )
            if self.config.is_encoder_decoder:
                cross_attentions += (outputs.cross_attentions,)

        if output_hidden_states:
            decoder_hidden_states += (
                (outputs.decoder_hidden_states,)
                if self.config.is_encoder_decoder
                else (outputs.hidden_states,)
            )

    # token selection
    if do_sample:
        probs = nn.functional.softmax(next_token_scores, dim=-1)
        # TODO (joao): this OP throws "skipping cudagraphs due to ['incompatible ops']", find solution
        next_tokens = torch.multinomial(probs, num_samples=1).squeeze(1)
    else:
        next_tokens = torch.argmax(next_token_scores, dim=-1)

    # finished sentences should have their next token be a padding token
    if has_eos_stopping_criteria:
        next_tokens = next_tokens * unfinished_sequences + pad_token_id * (1 - unfinished_sequences)

    # update generated ids, model inputs, and length for next step
    input_ids = torch.cat([input_ids, next_tokens[:, None]], dim=-1)
    if streamer is not None:
        streamer.put(next_tokens.cpu())

    unfinished_sequences = unfinished_sequences & ~stopping_criteria(input_ids, scores)
    this_peer_finished = unfinished_sequences.max() == 0
    
    # This is needed to properly delete outputs.logits which may be very large for first iteration
    # Otherwise a reference to outputs is kept which keeps the logits alive in the next iteration
    del outputs
```
其中 `outputs = model_forward(**model_inputs, return_dict=True)` 即为模型的单次推理调用，因此需要将性能采集函数嵌入到这段代码逻辑中。假设我们要采集 Decode 阶段的单次执行性能，则需要跳过 Prefill 阶段和首次 Decode 推理，因此设置 `schedule` 中的 `skip_first=2` 跳过 Prefill 和首次 Decode 推理，`wait` 和 `warmup` 设置为 0，`active` 和 `repeat` 设置为 1，仅采集第二次推理的性能数据。
嵌入性能采集后的代码段如下：
```
import torch_npu
# 添加Profiling采集扩展配置参数，详细参数介绍可参考下文的参数说明
experimental_config = torch_npu.profiler._ExperimentalConfig(
    export_type=[
        torch_npu.profiler.ExportType.Text
        ],
    profiler_level=torch_npu.profiler.ProfilerLevel.Level2
)

# 添加Profiling采集基础配置参数，详细参数介绍可参考下文的参数说明
prof = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.CPU,
        torch_npu.profiler.ProfilerActivity.NPU
        ],
    schedule=torch_npu.profiler.schedule(wait=0, warmup=0, active=1, repeat=1, skip_first=2),
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler("./result"),
    with_stack=True,
    experimental_config=experimental_config)
prof.start()        
while self._has_unfinished_sequences(this_peer_finished, synced_gpus, device=input_ids.device):
    if prefill_consumed:
        next_sequence_length = 1 if model_kwargs["use_cache"] else None
        model_inputs = self.prepare_inputs_for_generation(
            input_ids, next_sequence_length=next_sequence_length, **model_kwargs
        )
        with self._optimize_model_for_decode():
            outputs = model_forward(**model_inputs, return_dict=True)
    prefill_consumed = True
    model_kwargs = self._update_model_kwargs_for_generation(
        outputs,
        model_kwargs,
        is_encoder_decoder=self.config.is_encoder_decoder,
    )
    if synced_gpus and this_peer_finished:
        continue

    # Copy is needed to avoid keeping a hanging ref to outputs.logits which may be very large for first iteration
    # (the clone itself is always small)
    next_token_logits = outputs.logits[:, -1, :].to(copy=True, dtype=torch.float32, device=input_ids.device)

    # pre-process distribution
    next_token_scores = logits_processor(input_ids, next_token_logits)

    # Store scores, attentions and hidden_states when required
    if return_dict_in_generate:
        if output_scores:
            scores += (next_token_scores,)
        if output_logits:
            raw_logits += (next_token_logits,)
        if output_attentions:
            decoder_attentions += (
                (outputs.decoder_attentions,) if self.config.is_encoder_decoder else (outputs.attentions,)
            )
            if self.config.is_encoder_decoder:
                cross_attentions += (outputs.cross_attentions,)

        if output_hidden_states:
            decoder_hidden_states += (
                (outputs.decoder_hidden_states,)
                if self.config.is_encoder_decoder
                else (outputs.hidden_states,)
            )

    # token selection
    if do_sample:
        probs = nn.functional.softmax(next_token_scores, dim=-1)
        # TODO (joao): this OP throws "skipping cudagraphs due to ['incompatible ops']", find solution
        next_tokens = torch.multinomial(probs, num_samples=1).squeeze(1)
    else:
        next_tokens = torch.argmax(next_token_scores, dim=-1)

    # finished sentences should have their next token be a padding token
    if has_eos_stopping_criteria:
        next_tokens = next_tokens * unfinished_sequences + pad_token_id * (1 - unfinished_sequences)

    # update generated ids, model inputs, and length for next step
    input_ids = torch.cat([input_ids, next_tokens[:, None]], dim=-1)
    if streamer is not None:
        streamer.put(next_tokens.cpu())

    unfinished_sequences = unfinished_sequences & ~stopping_criteria(input_ids, scores)
    this_peer_finished = unfinished_sequences.max() == 0
    
    # This is needed to properly delete outputs.logits which may be very large for first iteration
    # Otherwise a reference to outputs is kept which keeps the logits alive in the next iteration
    del outputs
    prof.step()
prof.stop()
```
修改后的 `utils.py` 位于 src/qwen3.5 目录下，通过拷贝替换 transformers 中的 utils.py 文件完成修改。


In [ ]:
!cp src/qwen3.5/utils.py Sources/02/transformers/generation/utils.py 

---

### 4.3 执行推理脚本完成性能采集

大语言模型推理的主要步骤包括：

1. **加载模型和分词器**：使用transformers库加载预训练模型到NPU上
3. **准备输入文本**：构造对话格式的输入
4. **文本编码**：使用分词器将文本转换为模型输入格式并加载到NPU上
5. **模型推理**：调用generate方法生成回复
6. **结果解码**：将生成的token序列转换回文本


Qwen3.5-0.8B 完整推理代码如下，运行后将在 result 目录下生成性能数据：

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch_npu
model_name = "Sources/02/Qwen/" # 本地模型路径
model = AutoModelForCausalLM.from_pretrained(
    model_name
).half().npu()
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Hello!"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    top_k=20,
    top_p=1.00, 
    temperature= 1.0, 
    do_sample= True, 
    repetition_penalty= 1.0
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

运行完成后会生成 result 目录及性能数据文件。

---

## 5.性能数据查看与分析
将 result 目录下的性能数据文件下载到本地 PC 上，使用 MindStudio Insight 工具查看和分析性能数据。

### 5.1 MindStudio Insight安装

参考[《安装与卸载》](https://www.hiascend.com/document/detail/zh/mindstudio/840/GUI_baseddevelopmenttool/msascendinsightug/Insight_userguide_0005.html)

### 5.2 导入性能数据

<img src="./images/mindstudio_import_data.png" width="60%">

### 5.3 确认要优化的算子

分析需要优化的算子主要关注两点：1. 算子对模型性能的影响程度；2. 算子性能的可优化空间。

通过 MindStudio Insight 工具可以观察到每种类型算子的耗时占比，从而确定算子对模型性能的影响程度。

<img src="./images/mindstudio_operator_static.png" width="60%">

接下来分析耗时占比高的算子是否具有优化空间。Ascend C 算子开发系列教程中[第八章第二节《性能分析》“如何分析性能数据”](../../tutorials/ascendc_operator_development/08_performance_optimization/08.02_profiling_tool_usage.ipynb)中描述了如何分析算子的性能数据以判断是否存在优化空间，MindStudio Insight 工具中可以查看到分析所需的算子数据。由于同一类型的算子在不同输入 shape、dtype 和格式下性能各不相同，因此每种输入场景都需要单独分析，分析顺序可遵循耗时从高到低的顺序：

<img src="./images/mindstudio_operator_summary.png" width="60%"> 

### 5.4 确认待优化算子在Pytorch框架的代码位置

确定待优化的算子后，还需要定位该算子在 PyTorch 框架中的调用位置，以便开发完优化算子后进行替换。需要注意的是，优化算子可能并不会优化其在模型中的所有输入场景，因此需要根据优化的具体场景找到对应的 PyTorch 调用位置。以 Add 算子为例：

既然 Add 是要确认代码位置的算子，那么在 5.3 节中从 `Operator` 页签的 `See more` 一定可以了解到该算子的具体名称以及优化的输入场景信息（shape、dtype、format）。

<img src="./images/mindstudio_operator_add.png" width="60%"> 

获取这些信息后，在 `TimeLine` 页签中可以通过这些信息查找到该算子在时间线图中对应的执行块：

<img src="./images/mindstudio_timeline_add_search.png" width="60%"> 

点击执行块可以查看在 PyTorch 框架中的调用栈，从而定位算子的调用代码位置：

<img src="./images/mindstudio_timeline_add_pytorch.png" width="60%"> 

<img src="./images/mindstudio_timeline_add_stack.png" width="60%"> 


根据调用栈第一行，得到调用位置位于 `modeling_qwen3_5.py` 的第 437 行：

<img src="./images/mindstudio_add_call.jpg" width="60%"> 

---

## 课后实践

请根据本节课程学习内容完成以下题目进行自测。

1. （单选题）Ascend Pytorch Profiler 工具的核心作用是（）    
    A. 自动编写 Pytorch 框架代码    
    B. 采集分析 Pytorch 框架模型运行时的各类性能统计数据    
    C. Pytorch 模型功能测试工具    
    D. Pytorch 自动调优工具  

2. （单选题） Ascend Pytorch Profiler 工具可以使用参数控制采集第几次迭代的性能数据，请问是以下哪个参数（）   
    A. schedule  
    B. activities    
    C. on_trace_ready    
    D. with_stack    

3. （单选题）Ascend Pytorch Profiler 工具通过三个接口控制采集范围和迭代，以下哪个参数不是（）    
    A. start    
    B. end    
    C. stop    
    D. step    

4. （单选题）同一类型的算子性能受哪些因素影响（）  
    A. 输入shape  
    B. 输入dtype  
    C. 输入format  
    D. 以上都是    

5. （单选题）以下哪个不是大模型推理的步骤（）    
    A. 模型和分词器加载  
    B. 文本编码  
    C. 前向计算得到输出结果  
    D. 梯度计算与参数更新  

执行以下代码查看答案：

In [ ]:
!cat ./answers/02_answer.txt